In [7]:
"""
DIGITAL LENDING PLATFORM — SCORING ENGINE
Integrates Modules 5, 6, 7, 8, 10 into a unified inference pipeline.
"""
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import Optional
import hashlib, time, json
from datetime import datetime


# ── Risk Grade Thresholds ──────────────────────────────────────────────────
GRADE_THRESHOLDS = {
    "A": (0.00, 0.05),
    "B": (0.05, 0.10),
    "C": (0.10, 0.18),
    "D": (0.18, 0.30),
    "E": (0.30, 1.00),
}

LGD_BY_GRADE = {"A": 0.50, "B": 0.55, "C": 0.60, "D": 0.65, "E": 0.70}

EMPLOYMENT_INCOME_MEDIANS = {
    "Salaried":            45_000,
    "Self-Employed":       52_000,
    "Gig Worker":          22_000,
    "SME Owner":           85_000,
    "First-Time Borrower": 18_000,
}

# Interest rate floors/ceilings by grade
RATE_FLOORS = {"A": 10.5, "B": 12.5, "C": 15.0, "D": 18.0, "E": 22.0}
RATE_CEILS  = {"A": 14.0, "B": 16.5, "C": 20.0, "D": 24.0, "E": 32.0}


@dataclass
class ApplicationInput:
    # Demographics
    monthly_income: float
    employment_type: str
    credit_history_length: int  # months
    age: int

    # Loan request
    loan_amount: float
    loan_tenure_months: int
    loan_purpose: str = "Personal"

    # Credit signals
    credit_score: int = 650
    existing_loans: int = 0
    missed_payment_ratio: float = 0.05
    avg_delay_days: float = 5.0
    worst_delinquency_stage: int = 0   # 0=Current, 4=Write-Off

    # Financial stress
    financial_stress_index: float = 0.30
    debt_to_income_ratio: float = 2.0
    income_stability_score: float = 0.70

    # Behavioral
    spending_volatility_index: float = 0.25
    digital_engagement_score: float = 60.0
    bank_balance_avg: float = 50_000.0
    app_usage_mean: float = 15.0

    # Acquisition
    acquisition_channel: str = "Organic"
    city: str = "Mumbai"
    gender: str = "Not Specified"


@dataclass
class UnderwritingDecision:
    application_id: str
    timestamp: str

    # Risk scores
    pd_score: float
    risk_grade: str
    lgd: float
    expected_loss: float

    # Decision
    decision: str           # APPROVE / MANUAL_REVIEW / DECLINE
    confidence: float       # 0–1
    decision_rationale: str

    # Pricing
    interest_rate: float
    emi_amount: float
    approved_amount: float
    approved_tenure: int

    # Fraud
    fraud_risk_score: float
    fraud_severity: str

    # EWS
    health_score: float
    health_ladder: str
    delinquency_risk: float

    # XAI
    top_risk_factors: list = field(default_factory=list)
    protective_factors: list = field(default_factory=list)
    borrower_explanation: str = ""
    underwriter_notes: str = ""

    # Metadata
    processing_time_ms: float = 0.0


class ScoringEngine:
    """
    Unified scoring engine integrating PD, fraud, pricing, EWS, and XAI.
    Falls back to rule-based models when saved models are unavailable.
    """

    FEATURE_WEIGHTS = {
        # Repayment signals (highest predictive power)
        "missed_payment_ratio":     0.22,
        "worst_delinquency_stage":  0.20,
        "avg_delay_days":           0.10,
        # Financial stress
        "financial_stress_index":   0.15,
        "debt_to_income_ratio":     0.08,
        # Credit
        "credit_score":             0.12,  # inverse — higher score = lower risk
        "credit_history_length":    0.05,  # inverse — longer = lower risk
        # Behavioral
        "spending_volatility_index":0.08,
    }

    FRAUD_WEIGHTS = {
        "income_inflation_ratio":    0.35,
        "document_risk_proxy":       0.25,
        "identity_consistency":      0.20,  # inverse
        "synthetic_id_prob":         0.20,
    }

    def __init__(self, base_dir: Optional[str] = None):
        self.base_dir = base_dir
        self._try_load_models()

    def _try_load_models(self):
        """Attempt to load saved models; fall back to rule-based scoring."""
        self.model_loaded = False
        if self.base_dir:
            try:
                import joblib, os
                api_dir = os.path.join(self.base_dir, "explainable_ai", "apis")
                self.prod_model  = joblib.load(os.path.join(api_dir, "xai_prod_model.pkl"))
                self.preprocessor= joblib.load(os.path.join(api_dir, "xai_preprocessor.pkl"))
                self.features    = joblib.load(os.path.join(api_dir, "xai_features.pkl"))
                self.model_loaded= True
            except Exception:
                pass

    # ──────────────────────────────────────────────────────────────────────
    # PUBLIC API
    # ──────────────────────────────────────────────────────────────────────

    def score(self, app: ApplicationInput) -> UnderwritingDecision:
        t0 = time.time()
        app_id = self._generate_id(app)

        pd_score     = self._compute_pd(app)
        risk_grade   = self._grade(pd_score)
        lgd          = LGD_BY_GRADE[risk_grade]
        tenure_yr    = app.loan_tenure_months / 12
        cum_pd       = 1 - (1 - pd_score) ** tenure_yr
        expected_loss= round(cum_pd * lgd * app.loan_amount, 2)

        fraud_score, fraud_severity = self._compute_fraud(app)

        health_score, health_ladder = self._compute_health(app, pd_score)
        delinquency_risk = min(pd_score * 0.90, 0.99)

        rate = self._compute_rate(app, pd_score, risk_grade, health_score)
        approved_amount, approved_tenure = self._credit_decisioning(
            app, pd_score, fraud_score, risk_grade
        )
        emi = self._compute_emi(approved_amount, rate, approved_tenure)

        decision, confidence = self._make_decision(
            pd_score, fraud_score, risk_grade, app
        )

        risk_factors, protectives = self._explain(app, pd_score)
        borrower_text = self._borrower_narrative(
            decision, pd_score, risk_factors, protectives, app
        )
        underwriter_notes = self._underwriter_narrative(
            app, pd_score, risk_grade, fraud_score, expected_loss, rate
        )

        return UnderwritingDecision(
            application_id   = app_id,
            timestamp        = datetime.now().isoformat(),
            pd_score         = round(pd_score, 4),
            risk_grade       = risk_grade,
            lgd              = lgd,
            expected_loss    = expected_loss,
            decision         = decision,
            confidence       = round(confidence, 3),
            decision_rationale = self._decision_rationale(decision, risk_grade, pd_score, fraud_score),
            interest_rate    = round(rate, 2),
            emi_amount       = round(emi, 2),
            approved_amount  = round(approved_amount, 2),
            approved_tenure  = approved_tenure,
            fraud_risk_score = round(fraud_score, 4),
            fraud_severity   = fraud_severity,
            health_score     = round(health_score, 1),
            health_ladder    = health_ladder,
            delinquency_risk = round(delinquency_risk, 4),
            top_risk_factors = risk_factors,
            protective_factors = protectives,
            borrower_explanation = borrower_text,
            underwriter_notes  = underwriter_notes,
            processing_time_ms = round((time.time() - t0) * 1000, 1),
        )

    # ──────────────────────────────────────────────────────────────────────
    # INTERNAL SCORING METHODS
    # ──────────────────────────────────────────────────────────────────────

    def _compute_pd(self, app: ApplicationInput) -> float:
        """Rule-based PD with economic feature weights."""
        # Normalise each signal to [0, 1] risk contribution
        cs_norm  = max(0, (900 - app.credit_score) / 600)     # higher score = lower risk
        mpr_norm = min(app.missed_payment_ratio, 1.0)
        dpd_norm = app.worst_delinquency_stage / 4.0
        dly_norm = min(app.avg_delay_days / 90, 1.0)
        fsi_norm = min(app.financial_stress_index, 1.0)
        dti_norm = min(app.debt_to_income_ratio / 10, 1.0)
        svi_norm = min(app.spending_volatility_index, 1.0)
        hist_norm= max(0, 1 - app.credit_history_length / 60)  # longer = lower risk

        raw_pd = (
            0.22 * mpr_norm
            + 0.20 * dpd_norm
            + 0.12 * cs_norm
            + 0.15 * fsi_norm
            + 0.10 * dly_norm
            + 0.08 * dti_norm
            + 0.08 * svi_norm
            + 0.05 * hist_norm
        )

        # Thin-file adjustment
        if app.credit_history_length < 6:
            raw_pd = min(raw_pd + 0.06, 0.95)

        # Severe delinquency hard bump
        if app.worst_delinquency_stage >= 3:
            raw_pd = min(raw_pd + 0.15, 0.95)

        return float(np.clip(raw_pd, 0.01, 0.95))

    def _grade(self, pd_score: float) -> str:
        for g, (lo, hi) in GRADE_THRESHOLDS.items():
            if lo <= pd_score < hi:
                return g
        return "E"

    def _compute_fraud(self, app: ApplicationInput) -> tuple:
        """Compute composite fraud risk score."""
        exp_income = EMPLOYMENT_INCOME_MEDIANS.get(app.employment_type, 40_000)
        income_inflation = app.monthly_income / max(exp_income, 1)
        inflation_score  = min(max(income_inflation - 1.0, 0) / 2.0, 1.0)

        # Document/identity risk proxy
        doc_risk = float(
            (app.credit_history_length < 6) * 0.30
            + (app.worst_delinquency_stage >= 3) * 0.20
            + (1 - app.income_stability_score) * 0.25
        )

        identity_risk = 1 - min(
            0.90
            + (app.income_stability_score - 0.5) * 0.20
            - (app.credit_history_length < 6) * 0.20,
            1.0
        )

        synthetic_id = float(
            (app.credit_history_length < 6) * 0.40
            + (app.loan_amount > app.monthly_income * 20) * 0.25
            + doc_risk * 0.20
        )

        fraud_score = (
            0.35 * inflation_score
            + 0.25 * doc_risk
            + 0.25 * identity_risk
            + 0.15 * synthetic_id
        )
        fraud_score = float(np.clip(fraud_score, 0, 1))

        if fraud_score >= 0.75:   severity = "Red"
        elif fraud_score >= 0.55: severity = "Orange"
        elif fraud_score >= 0.35: severity = "Yellow"
        else:                     severity = "Green"

        return fraud_score, severity

    def _compute_health(self, app: ApplicationInput, pd_score: float) -> tuple:
        """Compute borrower health score (0-100)."""
        rep_h = (
            0.40 * (1 - app.missed_payment_ratio)
            - 0.30 * (app.avg_delay_days / 90)
            - 0.20 * (app.worst_delinquency_stage / 4)
            + 0.40
        )
        fin_h = (
            0.30 * ((app.credit_score - 300) / 600)
            + 0.30 * app.income_stability_score
            - 0.25 * app.financial_stress_index
            - 0.15 * min(app.debt_to_income_ratio / 10, 1)
            + 0.30
        )
        beh_h = (
            - 0.25 * app.spending_volatility_index
            + 0.15 * min(app.app_usage_mean / 30, 1)
            + 0.70
        )
        eng_h = app.digital_engagement_score / 100

        raw = (0.35 * rep_h + 0.25 * fin_h + 0.25 * beh_h + 0.15 * eng_h)
        score = float(np.clip(raw * 100, 5, 100))

        if score >= 70:   ladder = "Green"
        elif score >= 50: ladder = "Yellow"
        elif score >= 30: ladder = "Orange"
        else:             ladder = "Red"

        return score, ladder

    def _compute_rate(self, app: ApplicationInput, pd_score: float,
                      grade: str, health_score: float) -> float:
        """Risk-based pricing using Module 6 logic."""
        base = 8.50 + 2.00 + 2.00   # CoF + ops + margin
        risk_premium = pd_score * 0.55 * 100   # EL-adjusted
        beh_adj = -0.8 if health_score >= 70 else (0.5 if health_score < 50 else 0.0)

        rate = base + risk_premium + beh_adj
        rate = float(np.clip(rate, RATE_FLOORS[grade], RATE_CEILS[grade]))
        return rate

    def _credit_decisioning(self, app: ApplicationInput, pd_score: float,
                             fraud_score: float, grade: str) -> tuple:
        """Determine approved amount and tenure."""
        # Max exposure by grade
        grade_caps = {"A": 1.00, "B": 0.90, "C": 0.80, "D": 0.65, "E": 0.45}
        cap = grade_caps[grade]

        # Affordability: max EMI = 40% of monthly income
        max_emi_ratio = 0.40 if grade in ("A", "B") else 0.35
        max_emi       = app.monthly_income * max_emi_ratio
        # Back-solve max loan at requested rate
        rate = self._compute_rate(app, pd_score, grade, 60)
        mr   = rate / 100 / 12
        n    = app.loan_tenure_months
        if mr > 0:
            max_afford = max_emi * ((1 + mr)**n - 1) / (mr * (1 + mr)**n)
        else:
            max_afford = max_emi * n

        approved_amount = min(app.loan_amount, max_afford * cap)
        approved_amount = max(approved_amount, 0)
        approved_tenure = app.loan_tenure_months

        # Fraud adjustment
        if fraud_score > 0.55:
            approved_amount *= 0.60

        return approved_amount, approved_tenure

    def _compute_emi(self, principal: float, rate_pct: float, months: int) -> float:
        if principal <= 0 or months <= 0:
            return 0.0
        mr = rate_pct / 100 / 12
        if mr == 0:
            return principal / months
        return principal * mr * (1 + mr)**months / ((1 + mr)**months - 1)

    def _make_decision(self, pd_score: float, fraud_score: float,
                       grade: str, app: ApplicationInput) -> tuple:
        """Final approval decision and confidence."""
        # Hard decline rules
        if fraud_score > 0.75:
            return "DECLINE", 0.95
        if pd_score > 0.55:
            return "DECLINE", 0.90
        if app.worst_delinquency_stage >= 4:
            return "DECLINE", 0.99
        if app.debt_to_income_ratio > 7:
            return "DECLINE", 0.88

        # Manual review triggers
        if grade == "E" or (0.30 <= pd_score <= 0.55):
            return "MANUAL_REVIEW", 0.70
        if fraud_score > 0.45:
            return "MANUAL_REVIEW", 0.65
        if app.credit_history_length < 6:
            return "MANUAL_REVIEW", 0.60

        # Approval
        confidence = float(np.clip(1 - pd_score * 2, 0.55, 0.99))
        return "APPROVE", confidence

    def _explain(self, app: ApplicationInput, pd_score: float) -> tuple:
        """Generate top risk factors and protective signals."""
        signals = {
            "credit_score":            (900 - app.credit_score) / 600,
            "missed_payment_ratio":    app.missed_payment_ratio,
            "worst_delinquency_stage": app.worst_delinquency_stage / 4,
            "financial_stress_index":  app.financial_stress_index,
            "debt_to_income_ratio":    min(app.debt_to_income_ratio / 10, 1),
            "spending_volatility":     app.spending_volatility_index,
            "avg_delay_days":          min(app.avg_delay_days / 90, 1),
        }
        protects = {
            "income_stability_score":  app.income_stability_score,
            "digital_engagement":      app.digital_engagement_score / 100,
            "credit_history":          min(app.credit_history_length / 60, 1),
        }

        NARRATIVES = {
            "credit_score":            "Low credit score indicates elevated default risk",
            "missed_payment_ratio":    "History of missed payments — repayment reliability concern",
            "worst_delinquency_stage": "Prior delinquency on record",
            "financial_stress_index":  "High financial stress signals",
            "debt_to_income_ratio":    "High debt burden relative to income",
            "spending_volatility":     "Volatile spending behavior — financial instability signal",
            "avg_delay_days":          "Frequent payment delays",
            "income_stability_score":  "Stable, consistent income profile",
            "digital_engagement":      "Strong digital engagement — behavioral trust signal",
            "credit_history":          "Established credit history",
        }

        risk_factors = sorted(
            [{"feature": k, "score": round(v, 3), "description": NARRATIVES.get(k, k)}
             for k, v in signals.items() if v > 0.30],
            key=lambda x: x["score"], reverse=True
        )[:4]

        prot_factors = sorted(
            [{"feature": k, "score": round(v, 3), "description": NARRATIVES.get(k, k)}
             for k, v in protects.items() if v > 0.50],
            key=lambda x: x["score"], reverse=True
        )[:3]

        return risk_factors, prot_factors

    def _borrower_narrative(self, decision, pd_score, risk_factors,
                             protects, app) -> str:
        risk_text = "; ".join(r["description"] for r in risk_factors[:2]) or "standard risk profile"
        prot_text = "; ".join(p["description"] for p in protects[:2])

        if decision == "APPROVE":
            return (
                f"Congratulations! Your loan application has been approved. "
                f"Your application demonstrates a strong credit profile. "
                + (f"Key strengths: {prot_text}." if prot_text else "")
            )
        elif decision == "MANUAL_REVIEW":
            return (
                f"Your application is under additional review by our credit team. "
                f"A credit officer will contact you within 2 working days. "
                f"Some areas require further verification: {risk_text}."
            )
        else:
            return (
                f"We were unable to approve your application at this time. "
                f"Primary reasons: {risk_text}. "
                f"To improve eligibility: maintain on-time payments for 6+ months, "
                f"reduce outstanding debt obligations, and reapply after 90 days."
            )

    def _underwriter_narrative(self, app, pd_score, grade, fraud_score,
                                expected_loss, rate) -> str:
        return (
            f"PD: {pd_score:.2%} | Grade: {grade} | ECL: ₹{expected_loss:,.0f} | "
            f"Rate: {rate:.1f}% | Fraud: {fraud_score:.3f} | "
            f"DTI: {app.debt_to_income_ratio:.1f}x | "
            f"DPD Stage: {app.worst_delinquency_stage} | "
            f"Delinquency Hist: {app.missed_payment_ratio:.0%}"
        )

    def _decision_rationale(self, decision, grade, pd_score, fraud_score) -> str:
        if decision == "APPROVE":
            return f"Grade {grade} borrower. PD {pd_score:.2%} within appetite. Fraud risk acceptable."
        elif decision == "MANUAL_REVIEW":
            return f"Borderline Grade {grade}. PD {pd_score:.2%} or fraud {fraud_score:.3f} requires review."
        else:
            primary = []
            if pd_score > 0.30: primary.append(f"PD {pd_score:.2%} exceeds 30% limit")
            if fraud_score > 0.55: primary.append(f"Fraud score {fraud_score:.3f} critical")
            return "; ".join(primary) or "Does not meet minimum underwriting criteria"

    def _generate_id(self, app: ApplicationInput) -> str:
        seed = f"{app.monthly_income}{app.loan_amount}{time.time()}"
        return "APP" + hashlib.md5(seed.encode()).hexdigest()[:10].upper()

    def repayment_schedule(self, principal: float, rate_pct: float,
                            months: int) -> pd.DataFrame:
        """Generate full amortization table."""
        mr  = rate_pct / 100 / 12
        if mr == 0:
            emi = principal / months
        else:
            emi = principal * mr * (1 + mr)**months / ((1 + mr)**months - 1)

        records = []
        balance = principal
        for m in range(1, months + 1):
            interest   = balance * mr
            principal_p= emi - interest
            balance    = max(balance - principal_p, 0)
            records.append({
                "Month": m,
                "EMI (₹)": round(emi, 2),
                "Principal (₹)": round(principal_p, 2),
                "Interest (₹)": round(interest, 2),
                "Balance (₹)": round(balance, 2),
            })
        return pd.DataFrame(records)